In [70]:
import pandas as pd
#!pip3 install pickle5
import pickle5 as pickle
import os
def load_pickle(pickle_path):
    with open(pickle_path, 'rb') as f:
        pkl_file = pickle.load(f)
    return pkl_file

%cd /home/jupyter/MRI-spleen/predictCAD

/home/jupyter/MRI-spleen/predictCAD


In [147]:
cohort_name = 'cohorts/CADcohort_all_demo_rad.csv'
feats_of_interest = ['spleen_original_glszm_GrayLevelNonUniformity']#['spleen_original_firstorder_Kurtosis', 'spleen_original_gldm_LargeDependenceHighGrayLevelEmphasis', 
                  #'spleen_original_glszm_GrayLevelNonUniformity', 'spleen_original_glszm_SmallAreaLowGrayLevelEmphasis',
                   # 'spleen_original_glcm_Id']#, 'spleen_original_shape_Sphericity']
seg_fname = '2_wat.nii.gz' 
stitch_fname = 'wat.nii.gz'

In [157]:
coh = pd.read_csv(cohort_name)
print(coh.shape)
print(len(coh.ID.unique()))
#coh = coh[['ID']]
data = pd.read_csv("gs://ukbb_spleen/CHIPCAD_pheno_v5.csv", sep='\s+')
data = data[['ID', 'dm2_prev', 'antihtnbase']]
data = data.drop_duplicates('ID')
data = coh.merge(data, on = "ID", how = "inner")
print('ID' in coh.columns)
data.head()
print(data.shape)


(42543, 121)
42543
True
(42543, 123)


In [161]:
print(coh.columns)
#print(data.shape)
for elem in ['antihtnbase']:
    print(elem)
    print(data[elem].value_counts(dropna=False))
    #print(data[elem].mean())
    #print(data[elem].quantile([0.25, 0.75]))


Index(['age', 'spleen_original_shape_Elongation',
       'spleen_original_shape_Flatness',
       'spleen_original_shape_LeastAxisLength',
       'spleen_original_shape_MajorAxisLength',
       'spleen_original_shape_Maximum2DDiameterColumn',
       'spleen_original_shape_Maximum2DDiameterRow',
       'spleen_original_shape_Maximum2DDiameterSlice',
       'spleen_original_shape_Maximum3DDiameter',
       'spleen_original_shape_MeshVolume',
       ...
       'time_to_follow_up', 'Years_To_Coronary_Artery_Disease_INTERMEDIATE',
       'race_asian', 'race_black', 'race_mixed', 'race_other', 'race_white',
       'sex_Female', 'sex_Male', 'train'],
      dtype='object', length=121)
antihtnbase
0    38093
1     4450
Name: antihtnbase, dtype: int64


In [62]:
feat_to_lows = {}
feat_to_highs = {}
for feat in feats_of_interest:
    coh = coh.sort_values(by=[feat], ascending = True)
    feat_to_lows[feat] = list(coh.ID)[50:52]
    feat_to_highs[feat] = list(coh.ID)[-52:-50]
    

In [63]:
stitched_dir = '/home/jupyter/MRI-spleen/stitch_segment/stitched_data'
nnunet_folder = '/home/jupyter/MRI-spleen/stitch_segment/nnunet_data2'

orig_conversion_map = load_pickle(os.path.join(nnunet_folder, 'nnunet_data_conversion.pkl'))
conversion_map = {}
for entry in orig_conversion_map:
    entry = entry[0]
    for img_path in entry['img_paths']:
        conversion_map[img_path['orig']] = img_path['nnunet'] 
        #print(img_path['orig'])
        #break
print(len(conversion_map))        
#orig_conversion_map2 = load_pickle(os.path.join(nnunet_folder+'2', 'conversion.pkl'))
#conversion_map2 = {}
#for entry in orig_conversion_map2:
#    entry = entry[0]
#    for img_path in entry['img_paths']:
#        conversion_map2[img_path['orig']] = img_path['nnunet'] 
        #print(img_path['orig'])

173264


In [64]:
print(feat_to_lows)
print(feat_to_highs)

{'spleen_original_glszm_GrayLevelNonUniformity': [4974232, 3368445]}
{'spleen_original_glszm_GrayLevelNonUniformity': [4703925, 3058802]}


In [65]:
#%mkdir images
%cd /home/jupyter/MRI-spleen/predictCAD/images
import SimpleITK as sitk
import nibabel as nib
import numpy as np

/home/jupyter/MRI-spleen/predictCAD/images


In [66]:
for feat in feat_to_lows:
    fold_name = feat.replace('spleen_original_', '')+'2'
    %mkdir "$fold_name"
    %cd "$fold_name"
    print(fold_name)
    for ID in feat_to_lows[feat]:
        print(ID)
        from_seg_fname = '/home/jupyter/MRI-spleen/stitch_segment/segments/'+str(ID)+'/' + seg_fname
        to_seg_fname = str(ID)+'_low_'+seg_fname
        sitk_t1 = sitk.ReadImage(from_seg_fname)
        # and access the numpy array:
        t1 = sitk.GetArrayFromImage(sitk_t1)
        t1 = np.flipud(t1)
        t1_nifti = nib.Nifti1Image(t1, affine=np.eye(4), dtype = float)
        nib.save(t1_nifti, to_seg_fname)
        %cp "$from_seg_fname" "$to_seg_fname"
        
        stitched_ext = '/home/jupyter/MRI-spleen/stitched_data/'+str(ID)+'/'+stitch_fname
        from_stitched = conversion_map[stitched_ext]
        #print(from_stitched)
        frompath = 'gs://ukbb_spleen/nnunet_data/'+from_stitched.split("/")[-1]
        print(frompath)
        topath = str(ID)+'_low_'+stitch_fname
        !gsutil -m cp "$frompath" "$topath"
        
    for ID in feat_to_highs[feat]:
        print(ID)
        from_seg_fname = '/home/jupyter/MRI-spleen/stitch_segment/segments/'+str(ID)+'/' + seg_fname
        to_seg_fname = str(ID)+'_high_'+seg_fname
        sitk_t1 = sitk.ReadImage(from_seg_fname)
        # and access the numpy array:
        t1 = sitk.GetArrayFromImage(sitk_t1)
        t1 = np.flipud(t1)
        t1_nifti = nib.Nifti1Image(t1, affine=np.eye(4), dtype = float)
        nib.save(t1_nifti, to_seg_fname)
        
        stitched_ext = '/home/jupyter/MRI-spleen/stitched_data/'+str(ID)+'/'+stitch_fname
        from_stitched = conversion_map[stitched_ext]
        frompath = 'gs://ukbb_spleen/nnunet_data/'+from_stitched.split("/")[-1]
        topath = str(ID)+'_high_'+stitch_fname
        !gsutil -m cp "$frompath" "$topath"

        #print(from_stitched) # go download this later from bucket
        #to_stitched = str(ID)+'_high_'+stitch_fname
        #%cp "$from_stitched" "$to_stitched"
        

    %cd ..

        

mkdir: cannot create directory ‘glszm_GrayLevelNonUniformity2’: File exists
/home/jupyter/MRI-spleen/predictCAD/images/glszm_GrayLevelNonUniformity2
glszm_GrayLevelNonUniformity2
4974232
gs://ukbb_spleen/nnunet_data/ukbb_4ch_34376_0000.nii.gz
Copying gs://ukbb_spleen/nnunet_data/ukbb_4ch_34376_0000.nii.gz...
/ [1/1 files][ 26.9 MiB/ 26.9 MiB] 100% Done                                    
Operation completed over 1 objects/26.9 MiB.                                     
3368445
gs://ukbb_spleen/nnunet_data/ukbb_4ch_20554_0000.nii.gz
Copying gs://ukbb_spleen/nnunet_data/ukbb_4ch_20554_0000.nii.gz...
/ [1/1 files][ 28.4 MiB/ 28.4 MiB] 100% Done                                    
Operation completed over 1 objects/28.4 MiB.                                     
4703925
Copying gs://ukbb_spleen/nnunet_data/ukbb_4ch_32115_0000.nii.gz...
- [1/1 files][ 27.8 MiB/ 27.8 MiB] 100% Done                                    
Operation completed over 1 objects/27.8 MiB.                                 

[]